In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/new-raw-mooccubex/clean_raw_data.csv
/kaggle/input/new-raw-mooccubex/clean_data_mean.csv
/kaggle/input/new-raw-mooccubex/raw_data.csv
/kaggle/input/new-raw-mooccubex/clean_data_GCN.csv
/kaggle/input/new-raw-mooccubex/FillZero_minmax_baseline/clean_data_minmax_fill-zero.csv
/kaggle/input/new-raw-mooccubex/FillZero_minmax_baseline/clean_week2/val/val_week1_2.csv
/kaggle/input/new-raw-mooccubex/FillZero_minmax_baseline/clean_week2/test/test_week2.csv
/kaggle/input/new-raw-mooccubex/FillZero_minmax_baseline/clean_week2/train/clean_data_week2.csv
/kaggle/input/new-raw-mooccubex/FillZero_minmax_baseline/clean_week2/train/5-folds/data_part_2.csv
/kaggle/input/new-raw-mooccubex/FillZero_minmax_baseline/clean_week2/train/5-folds/data_part_3.csv
/kaggle/input/new-raw-mooccubex/FillZero_minmax_baseline/clean_week2/train/5-folds/data_part_4.csv
/kaggle/input/new-raw-mooccubex/FillZero_minmax_baseline/clean_week2/train/5-folds/data_part_1.csv
/kaggle/input/new-raw-mooccubex/FillZero_m

In [2]:
!pip uninstall -y scikit-learn imbalanced-learn
!pip install scikit-learn==1.2.2 imbalanced-learn==0.11.0

Found existing installation: scikit-learn 1.2.2
Uninstalling scikit-learn-1.2.2:
  Successfully uninstalled scikit-learn-1.2.2
Found existing installation: imbalanced-learn 0.13.0
Uninstalling imbalanced-learn-0.13.0:
  Successfully uninstalled imbalanced-learn-0.13.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.6/235.6 kB 13.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
mlxtend 0.23.4 requires scikit-learn>=1.3.1, but you have scikit-learn 1.2.2 which is incompatible.


In [3]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import backend as K
from kerastuner.tuners import RandomSearch
from sklearn.model_selection import StratifiedKFold
import time
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support, roc_auc_score

2025-09-05 03:28:29.371666: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757042909.703493      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757042909.791900      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
/tmp/ipykernel_19/3917850047.py:7: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  from kerastuner.tuners import RandomSearch


In [4]:
# Biến global cho base path
BASE_PATH = "/kaggle/input/new-raw-mooccubex/MLP_Adversarial_minmax_baseline"
# Tuần và số phần fold
weeks = ['week1', 'week2', 'week3', 'week4']
fold_parts = 5

# Tạo five_fold_files
five_fold_files = {
    week: [
        f"{BASE_PATH}/clean_{week}/train/5-folds/data_part_{i}.csv"
        for i in range(1, fold_parts + 1)
    ]
    for week in weeks
}

# Tạo file_validation
file_validation = {
    'week1': [f"{BASE_PATH}/clean_week1/val/val_week1.csv"],
    'week2': [f"{BASE_PATH}/clean_week2/val/val_week1_2.csv"],
    'week3': [f"{BASE_PATH}/clean_week3/val/val_week1_2_3.csv"],
    'week4': [f"{BASE_PATH}/clean_week4/val/val_week1_2_3_4.csv"]
}

# Tạo file_test
file_test = {
    week: [f"{BASE_PATH}/clean_{week}/test/test_{week}.csv"]
    for week in weeks
}

## Tìm siêu tham số tốt nhất cho từng tuần

In [5]:
# Định nghĩa Focal Loss
def focal_loss(gamma=2., alpha=0.25):
    def focal_loss_fixed(y_true, y_pred):
        y_pred = K.clip(y_pred, K.epsilon(), 1. - K.epsilon())
        cross_entropy = -y_true * K.log(y_pred)
        loss = alpha * K.pow(1 - y_pred, gamma) * cross_entropy
        return K.sum(loss, axis=-1)
    return focal_loss_fixed

# Tạo hàm train cho từng tuần
def train_week_model(week_number, file_paths_train, file_validataion):
    # Đọc dữ liệu
    train_data = pd.read_csv(file_paths_train)
    val_data = pd.read_csv(file_validataion)
    
    # Tách đặc trưng và nhãn
    X_train = train_data.drop(columns=["classification_encoded", "user_id", "course_id", "school", "enroll_time", "classification"])
    y_train = train_data["classification_encoded"]

    X_val = val_data.drop(columns=["classification_encoded", "user_id", "course_id", "school", "enroll_time", "classification"])
    y_val = val_data["classification_encoded"]
    
    # Áp dụng Over-sampling cho dữ liệu huấn luyện bằng SMOTE
    oversampler = SMOTE(sampling_strategy='auto', random_state=42)
    X_train_res, y_train_res = oversampler.fit_resample(X_train, y_train)
    
    # Reshape dữ liệu cho mô hình BiLSTM
    X_train_res = X_train_res.values.reshape(X_train_res.shape[0], X_train_res.shape[1], 1)
    X_val = X_val.values.reshape(X_val.shape[0], X_val.shape[1], 1)
    
    # One-hot encode nhãn
    y_train_res = tf.keras.utils.to_categorical(y_train_res, num_classes=5)
    y_val = tf.keras.utils.to_categorical(y_val, num_classes=5)
    
    def build_model(hp):
        inputs = tf.keras.Input(shape=(X_train_res.shape[1], 1))  # Khởi tạo đầu vào
        
        # RNN layer 1
        x = layers.SimpleRNN(
            units=hp.Int('units_1', min_value=32, max_value=256, step=32),
            return_sequences=True
        )(inputs)
        x = layers.Dropout(rate=hp.Float('dropout_1', min_value=0.1, max_value=0.5, step=0.1))(x)
        
        # RNN layer 2
        x = layers.SimpleRNN(
            units=hp.Int('units_2', min_value=32, max_value=256, step=32),
            return_sequences=False
        )(x)
        x = layers.Dropout(rate=hp.Float('dropout_2', min_value=0.1, max_value=0.5, step=0.1))(x)
        
        # Lớp đầu ra
        outputs = layers.Dense(5, activation='softmax')(x)
        
        # Khởi tạo mô hình
        model = tf.keras.Model(inputs=inputs, outputs=outputs)
        
        # Compile với Focal Loss
        model.compile(optimizer=tf.keras.optimizers.Adam(
                          learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')),
                      loss=focal_loss(gamma=2., alpha=0.25),
                      metrics=['accuracy'])
        
        return model

    
    # Khởi tạo RandomSearch tuner
    tuner = RandomSearch(
        build_model,
        objective='val_accuracy',
        max_trials=10,
        executions_per_trial=1,
        directory='my_dir',
        project_name=f'rnn_tuning_week{week_number}'
    )
    
    # Tìm kiếm siêu tham số tốt nhất
    tuner.search(X_train_res, y_train_res,
                 epochs=20,
                 validation_data=(X_val, y_val),
                 batch_size=32)
    
    # Trả về kết quả tối ưu cho tuần
    best_params = tuner.get_best_hyperparameters(num_trials=1)[0]
    return best_params

In [6]:
# Định nghĩa đường dẫn đến dữ liệu cho từng tuần
file_paths_train = {
    week: f"{BASE_PATH}/clean_{week}/train/clean_data_{week}.csv"
    for week in weeks
}

# Định nghĩa file_validation theo quy luật riêng
file_validation = {
    f"week{idx + 1}": f"{BASE_PATH}/clean_week{idx + 1}/val/val_week{'_'.join(str(i) for i in range(1, idx + 2))}.csv"
    for idx in range(len(weeks))
}

In [7]:
# Tìm tham số tốt nhất cho từng tuần
best_params_week1 = train_week_model(1, file_paths_train["week1"], file_validation["week1"])
best_params_week2 = train_week_model(2, file_paths_train["week2"], file_validation["week2"])
best_params_week3 = train_week_model(3, file_paths_train["week3"], file_validation["week3"])
best_params_week4 = train_week_model(4, file_paths_train["week4"], file_validation["week4"])

# In thông tin chi tiết các tham số tối ưu
print("Best Parameters for Week 1:")
for param_name in best_params_week1.values.keys():
    print(f"{param_name}: {best_params_week1.get(param_name)}")

print("\nBest Parameters for Week 2:")
for param_name in best_params_week2.values.keys():
    print(f"{param_name}: {best_params_week2.get(param_name)}")

print("\nBest Parameters for Week 3:")
for param_name in best_params_week3.values.keys():
    print(f"{param_name}: {best_params_week3.get(param_name)}")

print("\nBest Parameters for Week 4:")
for param_name in best_params_week4.values.keys():
    print(f"{param_name}: {best_params_week4.get(param_name)}")


Trial 10 Complete [00h 02m 43s]
val_accuracy: 0.6113483905792236

Best val_accuracy So Far: 0.981086015701294
Total elapsed time: 00h 27m 39s
Best Parameters for Week 1:
units_1: 96
dropout_1: 0.1
units_2: 160
dropout_2: 0.4
learning_rate: 0.0009380326717229366

Best Parameters for Week 2:
units_1: 224
dropout_1: 0.1
units_2: 224
dropout_2: 0.1
learning_rate: 0.00014340224055865128

Best Parameters for Week 3:
units_1: 256
dropout_1: 0.30000000000000004
units_2: 224
dropout_2: 0.4
learning_rate: 0.0001278856524227568

Best Parameters for Week 4:
units_1: 256
dropout_1: 0.1
units_2: 96
dropout_2: 0.4
learning_rate: 0.00016591955924639046


## Danh sách tham số tốt nhất của từng tuần

In [8]:
# Danh sách tham số tốt nhất
best_params = {
    "week1": best_params_week1,
    "week2": best_params_week2,
    "week3": best_params_week3,
    "week4": best_params_week4
}

In [9]:
from tensorflow.keras import layers
import tensorflow as tf
from tensorflow.keras import backend as K

# Định nghĩa Focal Loss
def focal_loss(gamma=2., alpha=0.25):
    def focal_loss_fixed(y_true, y_pred):
        y_pred = K.clip(y_pred, K.epsilon(), 1. - K.epsilon())
        cross_entropy = -y_true * K.log(y_pred)
        loss = alpha * K.pow(1 - y_pred, gamma) * cross_entropy
        return K.sum(loss, axis=-1)
    
    return focal_loss_fixed

# Xây dựng mô hình BiLSTM
def build_RNN_model(params, input_shape):
    inputs = tf.keras.Input(shape=input_shape)
    
    # RNN layer 1
    x = layers.SimpleRNN(
        units=params.get('units_1'),
        return_sequences=True
    )(inputs)
    x = layers.Dropout(rate=params.get('dropout_1', 0.2))(x)
    
    # RNN layer 2
    x = layers.SimpleRNN(
        units=params.get('units_2', 32),
        return_sequences=False
    )(x)
    x = layers.Dropout(rate=params.get('dropout_2', 0.2))(x)
    
    # Lớp đầu ra
    outputs = layers.Dense(5, activation='softmax')(x)
    
    # Khởi tạo mô hình
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    
    # Compile với Focal Loss
    model.compile(optimizer=tf.keras.optimizers.Adam(
                      learning_rate=params['learning_rate']),
                  loss=focal_loss(gamma=params.get('gamma', 2.), alpha=params.get('alpha', 0.25)),
                  metrics=['accuracy'])
    
    return model


In [10]:
# Biến lưu kết quả tổng quát
overall_results_5folds = []

# Lặp qua từng tuần
for week, file_paths in five_fold_files.items():
    print(f"\nProcessing {week} with best parameters...")
    params = best_params[week].values
    print(f"best parameters for {week}: {params}")
    
    # Biến lưu kết quả cho từng tuần
    week_results = {
        "week": week,
        "accuracy_per_fold": [],
        "precision_per_label": [],
        "recall_per_label": [],
        "f1_score_per_label": [],
        "auc_roc_per_label": [],    # AUC từng lớp
        "auc_roc_macro": [],        # AUC macro
        "auc_roc_weighted": [],     # AUC weighted (tự tính)
        "precision_macro": [],
        "recall_macro": [],
        "f1_macro": [],
        "precision_weighted": [],
        "recall_weighted": [],
        "f1_weighted": [],
        "confusion_matrices": [],
        "train_times": [],
        "test_times": []
    }

    # Lặp qua từng fold
    for i in range(len(file_paths)):
        print(f"Fold {i+1}: Using file {file_paths[i]} as test set")
        
        # Tải dữ liệu
        test_data = pd.read_csv(file_paths[i])
        train_data = pd.concat([pd.read_csv(file_paths[j]) for j in range(len(file_paths)) if j != i])
        
        # Tách X và y
        X_train = train_data.drop(columns=["classification_encoded", "user_id",
                                           "course_id", "school", "enroll_time", "classification"])
        y_train = to_categorical(train_data['classification_encoded'], num_classes=5)
        
        X_test = test_data.drop(columns=["classification_encoded", "user_id",
                                         "course_id", "school", "enroll_time", "classification"])
        y_test = to_categorical(test_data['classification_encoded'], num_classes=5)

        # Reshape dữ liệu cho LSTM
        X_train = X_train.to_numpy().reshape((X_train.shape[0], 1, X_train.shape[1]))
        X_test = X_test.to_numpy().reshape((X_test.shape[0], 1, X_test.shape[1]))
        input_shape = (X_train.shape[1], X_train.shape[2])  # Lấy kích thước từ dữ liệu thực tế
        # Xây dựng mô hình với tham số tốt nhất
        model = build_RNN_model(params, input_shape)
        
        # Bắt đầu tính thời gian huấn luyện
        start_train = time.time()
        model.fit(X_train, y_train, epochs=20, validation_data=(X_test, y_test), batch_size=32)
        end_train = time.time()
        
        # Bắt đầu tính thời gian kiểm thử
        start_test = time.time()
        y_pred = model.predict(X_test)
        end_test = time.time()
        
        # Tính thời gian và lưu lại
        train_time = end_train - start_train
        test_time = end_test - start_test
        week_results["train_times"].append(train_time)
        week_results["test_times"].append(test_time)

        # Đánh giá mô hình trên tập kiểm thử của fold hiện tại
        _, accuracy = model.evaluate(X_test, y_test, verbose=0)
        week_results["accuracy_per_fold"].append(accuracy)
        
        # Dự đoán
        y_pred_classes = y_pred.argmax(axis=1)
        y_test_classes = y_test.argmax(axis=1)
        
        # Tính các chỉ số cho mỗi fold
        precision, recall, f1, _ = precision_recall_fscore_support(y_test_classes, y_pred_classes, average=None)
        precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(y_test_classes, y_pred_classes, average='macro')
        precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(y_test_classes, y_pred_classes, average='weighted')
        conf_matrix = confusion_matrix(y_test_classes, y_pred_classes)
        
        # Tính AUC-ROC
        try:
            # Tính AUC macro và theo từng lớp với OvR
            auc_macro = roc_auc_score(y_test, y_pred, multi_class="ovr", average="macro")
            auc_per_class = roc_auc_score(y_test, y_pred, multi_class="ovr", average=None)
            # Tính AUC weighted: tính trọng số theo số mẫu của từng lớp
            supports = np.bincount(y_test_classes, minlength=5)
            auc_weighted = np.sum(auc_per_class * supports) / np.sum(supports)
        except Exception as e:
            print(f"Lỗi khi tính AUC: {e}")
            auc_macro = np.nan
            auc_per_class = [np.nan] * 5
            auc_weighted = np.nan
            
        # Lưu kết quả của fold hiện tại
        week_results["precision_per_label"].append(precision)
        week_results["recall_per_label"].append(recall)
        week_results["f1_score_per_label"].append(f1)
        week_results["auc_roc_per_label"].append(auc_per_class)  # AUC từng lớp
        week_results["auc_roc_macro"].append(auc_macro)          # AUC macro
        week_results["auc_roc_weighted"].append(auc_weighted)      # AUC weighted
        week_results["confusion_matrices"].append(conf_matrix)
        week_results["precision_macro"].append(precision_macro)
        week_results["recall_macro"].append(recall_macro)
        week_results["f1_macro"].append(f1_macro)
        week_results["precision_weighted"].append(precision_weighted)
        week_results["recall_weighted"].append(recall_weighted)
        week_results["f1_weighted"].append(f1_weighted)

    # Tính trung bình cho từng nhãn
    average_precision_per_label = np.mean(week_results["precision_per_label"], axis=0)
    average_recall_per_label = np.nanmean(week_results["recall_per_label"], axis=0)
    average_f1_per_label = np.nanmean(week_results["f1_score_per_label"], axis=0)
    average_auc_per_label = np.nanmean(week_results["auc_roc_per_label"], axis=0)
    average_confusion_matrix = np.nanmean(week_results["confusion_matrices"], axis=0)
    average_train_time = sum(week_results["train_times"]) / len(week_results["train_times"])
    average_test_time = sum(week_results["test_times"]) / len(week_results["test_times"])
    average_accuracy = np.nanmean(week_results["accuracy_per_fold"])
    average_precision_macro = np.nanmean(week_results["precision_macro"])
    average_recall_macro = np.nanmean(week_results["recall_macro"])
    average_f1_macro = np.nanmean(week_results["f1_macro"])
    average_auc_macro = np.nanmean(week_results["auc_roc_macro"])
    average_precision_weighted = np.nanmean(week_results["precision_weighted"])
    average_recall_weighted = np.nanmean(week_results["recall_weighted"])
    average_f1_weighted = np.nanmean(week_results["f1_weighted"])
    average_auc_weighted = np.nanmean(week_results["auc_roc_weighted"])


    # Tạo DataFrame cho precision, recall, f1-score
    labels = np.unique(y_test_classes)  # Lấy nhãn từ y_test_classes
    metrics_df = pd.DataFrame({
        "Label": labels,
        "Average Precision": average_precision_per_label,
        "Average Recall": average_recall_per_label,
        "Average F1-Score": average_f1_per_label,
        "Average AUC": average_auc_per_label
    })
    
    # Tạo DataFrame cho confusion matrix
    confusion_df = pd.DataFrame(average_confusion_matrix, index=labels, columns=labels)
    # In kết quả Accuracy và Macro metrics
    print("\n=== Average Accuracy ===")
    print(f"{average_accuracy:.4f}")
    print("\n=== Average Macro Metrics ===")
    print(f"Macro Precision: {average_precision_macro:.4f}")
    print(f"Macro Recall: {average_recall_macro:.4f}")
    print(f"Macro F1-Score: {average_f1_macro:.4f}")
    print(f"Macro AUC-ROC: {average_auc_macro:.4f}")
    print("\n=== Average Weighted Metrics ===")
    print(f"Weighted Precision: {average_precision_weighted:.4f}")
    print(f"Weighted Recall: {average_recall_weighted:.4f}")
    print(f"Weighted F1-Score: {average_f1_weighted:.4f}")
    print(f"Weighted AUC-ROC: {average_auc_weighted:.4f}")
    print("\n=== Average Metrics per Label ===")
    print(metrics_df)
    print("\n=== Average Confusion Matrix ===")
    print(confusion_df)
    
    # Cập nhật kết quả cho tuần hiện tại
    week_results.update({
        "average_accuracy": average_accuracy,
        "average_precision_macro": average_precision_macro,
        "average_recall_macro": average_recall_macro,
        "average_f1_macro": average_f1_macro,
        "average_auc_macro": average_auc_macro,
        "average_precision_weighted": average_precision_weighted,
        "average_recall_weighted": average_recall_weighted,
        "average_f1_weighted": average_f1_weighted,
        "average_auc_weighted": average_auc_weighted,
        "average_metrics_df": metrics_df,
        "average_confusion_matrix": confusion_df,
        "average_train_times": average_train_time,
        "average_test_times": average_test_time,
    })
    overall_results_5folds.append(week_results)


Processing week1 with best parameters...
best parameters for week1: {'units_1': 96, 'dropout_1': 0.1, 'units_2': 160, 'dropout_2': 0.4, 'learning_rate': 0.0009380326717229366}
Fold 1: Using file /kaggle/input/new-raw-mooccubex/MLP_Adversarial_minmax_baseline/clean_week1/train/5-folds/data_part_1.csv as test set
Epoch 1/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.5626 - loss: 0.1859 - val_accuracy: 0.6687 - val_loss: 0.1213
Epoch 2/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6339 - loss: 0.1372 - val_accuracy: 0.6817 - val_loss: 0.1187
Epoch 3/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6481 - loss: 0.1308 - val_accuracy: 0.6687 - val_loss: 0.1228
Epoch 4/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6611 - loss: 0.1225 - val_accuracy: 0.6839 - val_loss: 0.1127
Epoch 5/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6679 - loss: 0.1231 - val_accuracy: 0.7175 - val_loss: 0.1077
Epoch 6/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3m

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


328/328 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.5708 - loss: 0.1792 - val_accuracy: 0.6992 - val_loss: 0.1191
Epoch 2/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6349 - loss: 0.1327 - val_accuracy: 0.6275 - val_loss: 0.1213
Epoch 3/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6653 - loss: 0.1222 - val_accuracy: 0.7083 - val_loss: 0.1128
Epoch 4/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6711 - loss: 0.1176 - val_accuracy: 0.6855 - val_loss: 0.1117
Epoch 5/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6903 - loss: 0.1125 - val_accuracy: 0.7228 - val_loss: 0.1042
Epoch 6/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7023 - loss: 0.1085 - val_accuracy: 0.7122 - val_loss: 0.1054
Epoch 7/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7010 - loss: 0.1086 - val_accuracy: 0.7499 - val_loss: 0.0975
Epoch 8/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7169 - loss: 0.1034 - val_accuracy: 0.7449 - val_

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


328/328 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.5852 - loss: 0.1679 - val_accuracy: 0.6644 - val_loss: 0.1192
Epoch 2/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6414 - loss: 0.1333 - val_accuracy: 0.6632 - val_loss: 0.1202
Epoch 3/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6676 - loss: 0.1251 - val_accuracy: 0.6678 - val_loss: 0.1133
Epoch 4/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6712 - loss: 0.1206 - val_accuracy: 0.6770 - val_loss: 0.1137
Epoch 5/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6824 - loss: 0.1160 - val_accuracy: 0.7048 - val_loss: 0.1086
Epoch 6/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6904 - loss: 0.1110 - val_accuracy: 0.6968 - val_loss: 0.1108
Epoch 7/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7066 - loss: 0.1099 - val_accuracy: 0.7101 - val_loss: 0.1065
Epoch 8/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7122 - loss: 0.1044 - val_accuracy: 0.7193 - val

...

Fold 4: Using file /kaggle/input/new-raw-mooccubex/MLP_Adversarial_minmax_baseline/clean_week3/train/5-folds/data_part_4.csv as test set
Epoch 1/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.5502 - loss: 0.2071 - val_accuracy: 0.6827 - val_loss: 0.1337
Epoch 2/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5953 - loss: 0.1721 - val_accuracy: 0.6873 - val_loss: 0.1234
Epoch 3/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6150 - loss: 0.1604 - val_accuracy: 0.6983 - val_loss: 0.1259
Epoch 4/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6334 - loss: 0.1476 - val_accuracy: 0.7151 - val_loss: 0.1133
Epoch 5/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6460 - loss: 0.1377 - val_accuracy: 0.7342 - val_loss: 0.1064
Epoch 6/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6694 - loss: 0.1308 - val_accuracy: 0.7418 - val_loss: 0.1063
Epoch 7/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6977 - loss: 0.1211 - val_acc

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Fold 5: Using file /kaggle/input/new-raw-mooccubex/MLP_Adversarial_minmax_baseline/clean_week3/train/5-folds/data_part_5.csv as test set
Epoch 1/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.5378 - loss: 0.2039 - val_accuracy: 0.6529 - val_loss: 0.1329
Epoch 2/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5810 - loss: 0.1751 - val_accuracy: 0.6785 - val_loss: 0.1292
Epoch 3/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6092 - loss: 0.1559 - val_accuracy: 0.6911 - val_loss: 0.1289
Epoch 4/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6167 - loss: 0.1536 - val_accuracy: 0.6983 - val_loss: 0.1175
Epoch 5/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6457 - loss: 0.1389 - val_accuracy: 0.7227 - val_loss: 0.1194
Epoch 6/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6660 - loss: 0.1320 - val_accuracy: 0.7330 - val_loss: 0.1017
Epoch 7/20
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7033 - loss: 0.1185 - val_acc

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



=== Average Accuracy ===
0.8441

=== Average Macro Metrics ===
Macro Precision: 0.5963
Macro Recall: 0.5502
Macro F1-Score: 0.5613
Macro AUC-ROC: 0.9326

=== Average Weighted Metrics ===
Weighted Precision: 0.8180
Weighted Recall: 0.8441
Weighted F1-Score: 0.8255
Weighted AUC-ROC: 0.9632

=== Average Metrics per Label ===
   Label  Average Precision  Average Recall  Average F1-Score  Average AUC
0      0           0.691005        0.856000          0.763546     0.951838
1      1           0.000000        0.000000          0.000000     0.895721
2      2           0.734440        0.487871          0.582515     0.914433
3      3           0.616375        0.443114          0.508963     0.920242
4      4           0.939836        0.964077          0.951659     0.980646

=== Average Confusion Matrix ===
       0    1     2     3       4
0  513.6  0.0  16.2  23.0    47.2
1   60.8  0.0   2.2   9.4    15.2
2   57.2  0.0  80.2   8.4    18.6
3   70.6  0.0   3.8  74.0    18.6
4   43.2  0.2   7.4  

## Kết quả cross validation trên 5-folds

In [11]:
# Duyệt qua các tuần trong overall_results
for week_result in overall_results_5folds:
    week = week_result["week"]
    average_train_time = np.mean(week_result["train_times"])
    average_test_time = np.mean(week_result["test_times"])
    average_metrics_df = week_result["average_metrics_df"]
    average_accuracy = np.mean(week_results["accuracy_per_fold"])
    average_confusion_matrix = week_result["average_confusion_matrix"]
    
    # In kết quả
    print(f"\n=== Results for {week} ===")
    print(f"Average Accurancy: {average_accuracy}")
    print(f"Average Train Time: {average_train_time:.4f} seconds")
    print(f"Average Test Time: {average_test_time:.4f} seconds")
    print(f"Average AUC Macro: {average_auc_macro}")
    print(f"Average AUC Weighted: {average_auc_weighted}")
    print("\nAverage Precision, Recall, F1-Score, AUC-ROC per Label:")
    print(average_metrics_df)
    print("\nAverage Confusion Matrix:")
    print(average_confusion_matrix)



=== Results for week1 ===
Average Accurancy: 0.8965823531150818
Average Train Time: 25.3699 seconds
Average Test Time: 1.3759 seconds
Average AUC Macro: 0.9645819165300719
Average AUC Weighted: 0.9831841330422517

Average Precision, Recall, F1-Score, AUC-ROC per Label:
   Label  Average Precision  Average Recall  Average F1-Score  Average AUC
0      0           0.610907        0.612667          0.604808     0.861575
1      1           0.000000        0.000000          0.000000     0.836437
2      2           0.639695        0.498692          0.549625     0.875176
3      3           0.612067        0.433533          0.500492     0.904825
4      4           0.836817        0.911801          0.870910     0.918456

Average Confusion Matrix:
       0    1     2     3       4
0  367.6  0.0  22.8  22.4   187.2
1   45.0  0.0   2.6   9.0    31.0
2   38.6  0.0  82.0   7.6    36.2
3   53.0  0.8   4.0  72.4    36.8
4  111.2  0.0  21.6   8.6  1462.0

=== Results for week2 ===
Average Accurancy: 0.

## Kiểm tra trên tập test

In [12]:
# Mảng lưu dữ liệu của các tuần
results = []

def process_week(week_num, best_params, results):
    print(f"\n=== Processing Week {week_num} ===")
    params = best_params[f"week{week_num}"].values
    # Đường dẫn tới dữ liệu tuần tương ứng
    train_path = f"{BASE_PATH}/clean_week{week_num}/train/clean_data_week{week_num}.csv"
    test_path = f"{BASE_PATH}/clean_week{week_num}/test/test_week{week_num}.csv"
    
    # Load dữ liệu
    train_data = pd.read_csv(train_path)
    test_data = pd.read_csv(test_path)
    
    # Tách X và y
    X_train = train_data.drop(columns=["classification_encoded", "user_id",
                                       "course_id", "school", "enroll_time", "classification"])
    y_train = train_data['classification_encoded']
    
    X_test = test_data.drop(columns=["classification_encoded", "user_id",
                                     "course_id", "school", "enroll_time", "classification"])
    y_test = test_data['classification_encoded']

    # Áp dụng SMOTE cho tập huấn luyện
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

    # Chuyển đổi nhãn sang dạng one-hot
    y_train_resampled = to_categorical(y_train_resampled, num_classes=5)
    y_test = to_categorical(y_test, num_classes=5)

    # Reshape dữ liệu cho LSTM
    X_train_resampled = X_train_resampled.to_numpy().reshape((X_train_resampled.shape[0], 1, X_train_resampled.shape[1]))
    X_test = X_test.to_numpy().reshape((X_test.shape[0], 1, X_test.shape[1]))
    
    # Xây dựng mô hình với tham số tốt nhất
    input_shape = (X_train_resampled.shape[1], X_train_resampled.shape[2])  # Lấy kích thước từ dữ liệu thực tế
    model = build_RNN_model(params, input_shape)
    
    # Huấn luyện mô hình
    start_train = time.time()
    model.fit(X_train_resampled, y_train_resampled, epochs=20, validation_split=0.1, batch_size=32)
    end_train = time.time()
    
    # Kiểm thử mô hình
    start_test = time.time()
    y_pred = model.predict(X_test)
    end_test = time.time()
    
    # Tính thời gian huấn luyện và kiểm thử
    train_time = end_train - start_train
    test_time = end_test - start_test
    
    # Đánh giá mô hình
    y_pred_classes = y_pred.argmax(axis=1)
    y_test_classes = y_test.argmax(axis=1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(y_test_classes, y_pred_classes, average=None)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(y_test_classes, y_pred_classes, average='macro')
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(y_test_classes, y_pred_classes, average='weighted')
    conf_matrix = confusion_matrix(y_test_classes, y_pred_classes)
    accuracy = accuracy_score(y_test_classes, y_pred_classes)
    
    # Tính AUC-ROC (với one-vs-rest)
    try:
        auc_macro = roc_auc_score(y_test, y_pred, multi_class="ovr", average="macro")
        auc_per_class = roc_auc_score(y_test, y_pred, multi_class="ovr", average=None)
        # Tính AUC weighted tự tính theo trọng số mẫu của từng lớp
        supports = np.bincount(y_test_classes, minlength=5)
        auc_weighted = np.sum(auc_per_class * supports) / np.sum(supports)
    except Exception as e:
        print(f"Lỗi khi tính AUC: {e}")
        auc_macro = np.nan
        auc_per_class = [np.nan] * 5
        auc_weighted = np.nan

    # Lưu kết quả vào mảng
    results.append({
        "week": week_num,
        "train_time": train_time,
        "test_time": test_time,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
        "auc_macro": auc_macro,
        "auc_weighted": auc_weighted,
        "auc_per_class": auc_per_class,
        "confusion_matrix": conf_matrix
    })
    
    # In kết quả chi tiết
    print("\n=== Precision, Recall, F1-Score per Label ===")
    print(pd.DataFrame({
        "Label": np.unique(y_test_classes),
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    }))

    print("\n=== Macro Averages & Accuracy ===")
    print(f"Macro Precision: {precision_macro:.4f}")
    print(f"Macro Recall: {recall_macro:.4f}")
    print(f"Macro F1-Score: {f1_macro:.4f}")
    print(f"Accuracy: {accuracy:.4f}")
    
    print("\n=== Weighted Averages ===")
    print(f"Weighted Precision: {precision_weighted:.4f}")
    print(f"Weighted Recall: {recall_weighted:.4f}")
    print(f"Weighted F1-Score: {f1_weighted:.4f}")
    
    print("\n=== AUC-ROC ===")
    print(f"AUC Macro: {auc_macro:.4f}")
    print(f"AUC Weighted: {auc_weighted:.4f}")
    print(f"AUC per Label: {auc_per_class}")
    
    print("\n=== Confusion Matrix ===")
    print(pd.DataFrame(conf_matrix, index=np.unique(y_test_classes), columns=np.unique(y_test_classes)))
    
    print(f"\nTrain Time: {train_time:.2f} seconds")
    print(f"Test Time: {test_time:.2f} seconds")

In [13]:
process_week(1, best_params, results)


=== Processing Week 1 ===
Epoch 1/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - accuracy: 0.4159 - loss: 0.2328 - val_accuracy: 0.5079 - val_loss: 0.2592
Epoch 2/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5332 - loss: 0.1784 - val_accuracy: 0.4410 - val_loss: 0.2430
Epoch 3/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5630 - loss: 0.1650 - val_accuracy: 0.4669 - val_loss: 0.2585
Epoch 4/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5807 - loss: 0.1597 - val_accuracy: 0.5138 - val_loss: 0.2462
Epoch 5/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5896 - loss: 0.1557 - val_accuracy: 0.5321 - val_loss: 0.2441
Epoch 6/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5954 - loss: 0.1535 - val_accuracy: 0.4360 - val_loss: 0.2575
Epoch 7/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6104 - loss: 0.1470 - val_accuracy: 0.4585 - val_loss: 0.2726
Epoch 8/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accura

In [14]:
process_week(2, best_params, results)


=== Processing Week 2 ===
Epoch 1/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.4337 - loss: 0.2163 - val_accuracy: 0.4403 - val_loss: 0.2579
Epoch 2/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5440 - loss: 0.1773 - val_accuracy: 0.6248 - val_loss: 0.1983
Epoch 3/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5829 - loss: 0.1552 - val_accuracy: 0.5415 - val_loss: 0.2133
Epoch 4/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6091 - loss: 0.1425 - val_accuracy: 0.6159 - val_loss: 0.2073
Epoch 5/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6247 - loss: 0.1348 - val_accuracy: 0.6044 - val_loss: 0.1960
Epoch 6/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6365 - loss: 0.1322 - val_accuracy: 0.4320 - val_loss: 0.3242
Epoch 7/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6529 - loss: 0.1266 - val_accuracy: 0.7004 - val_loss: 0.1502
Epoch 8/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accura

In [15]:
process_week(3, best_params, results)


=== Processing Week 3 ===
Epoch 1/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.3355 - loss: 0.2974 - val_accuracy: 0.4540 - val_loss: 0.3366
Epoch 2/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.4426 - loss: 0.2256 - val_accuracy: 0.4630 - val_loss: 0.2208
Epoch 3/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.4957 - loss: 0.1942 - val_accuracy: 0.4433 - val_loss: 0.2737
Epoch 4/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5440 - loss: 0.1693 - val_accuracy: 0.4719 - val_loss: 0.2828
Epoch 5/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5751 - loss: 0.1544 - val_accuracy: 0.4093 - val_loss: 0.3005
Epoch 6/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6089 - loss: 0.1441 - val_accuracy: 0.5298 - val_loss: 0.2236
Epoch 7/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6197 - loss: 0.1387 - val_accuracy: 0.5587 - val_loss: 0.2492
Epoch 8/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accura

In [16]:
process_week(4, best_params, results)


=== Processing Week 4 ===
Epoch 1/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.3736 - loss: 0.2613 - val_accuracy: 0.4413 - val_loss: 0.2923
Epoch 2/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5057 - loss: 0.1928 - val_accuracy: 0.4093 - val_loss: 0.3051
Epoch 3/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5938 - loss: 0.1522 - val_accuracy: 0.4006 - val_loss: 0.3010
Epoch 4/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6454 - loss: 0.1271 - val_accuracy: 0.6291 - val_loss: 0.1765
Epoch 5/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6991 - loss: 0.1085 - val_accuracy: 0.5173 - val_loss: 0.2312
Epoch 6/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7465 - loss: 0.0913 - val_accuracy: 0.5914 - val_loss: 0.1638
Epoch 7/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7829 - loss: 0.0767 - val_accuracy: 0.6987 - val_loss: 0.1004
Epoch 8/20
1128/1128 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accura

In [17]:
# Hiển thị dữ liệu của các tuần
print("\n=== Summary Results for All Weeks ===")
for result in results:
    print(f"Week {result['week']}:")
    print(f"  Train Time: {result['train_time']:.2f} seconds")
    print(f"  Test Time: {result['test_time']:.2f} seconds")
    print(f"  Accurancy: {result['accuracy']}")
    print(f"  Precision: {result['precision']}")
    print(f"  Recall: {result['recall']}")
    print(f"  F1-Score: {result['f1_score']}")
    print(f"  Macro Precision: {result['precision_macro']}")
    print(f"  Macro Recall: {result['recall_macro']}")
    print(f"  Macro F1-Score: {result['f1_macro']}")
    print(f"  Confusion Matrix:\n{result['confusion_matrix']}")
    print("\n=== AUC-ROC ===")
    print(f"AUC Macro: {result['auc_macro']:.4f}")
    print(f"AUC Weighted: {result['auc_weighted']:.4f}")
    print(f"AUC per Label: {result['auc_per_class']}")


=== Summary Results for All Weeks ===
Week 1:
  Train Time: 63.63 seconds
  Test Time: 1.31 seconds
  Accurancy: 0.7469512195121951
  Precision: [0.56440678 0.31460674 0.55670103 0.775      0.95408163]
  Recall: [0.888      0.51851852 0.52427184 0.59047619 0.74576271]
  F1-Score: [0.69015544 0.39160839 0.54       0.67027027 0.83715725]
  Macro Precision: 0.6329592369629895
  Macro Recall: 0.6534058531038619
  Macro F1-Score: 0.6258382698150973
  Confusion Matrix:
[[333  17   9   2  14]
 [ 17  28   3   1   5]
 [ 30  10  54   2   7]
 [ 11  13   9  62  10]
 [199  21  22  13 748]]

=== AUC-ROC ===
AUC Macro: 0.9285
AUC Weighted: 0.9354
AUC per Label: [0.90765744 0.92368409 0.91792737 0.9461517  0.94699262]
Week 2:
  Train Time: 61.29 seconds
  Test Time: 1.27 seconds
  Accurancy: 0.8853658536585366
  Precision: [0.97452229 0.5875     0.49152542 0.70535714 0.97492163]
  Recall: [0.816      0.87037037 0.84466019 0.75238095 0.93020937]
  F1-Score: [0.88824383 0.70149254 0.62142857 0.7281106 